In [1]:
# Importing libraries:
import psycopg2
import pandas as pd
pd.set_option('display.max_rows', 5) 
from sqlalchemy import create_engine
import logging
logging.getLogger('sqlalchemy.engine').setLevel(logging.WARNING)
import re
import gc
import time
import numpy as np
import warnings
import uuid
from datetime import datetime, timedelta
import ast
from io import StringIO

# Setting up logging configuration
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


# ============================================================================
# SECTION 1: DATABASE CONNECTION
# ============================================================================

def create_connection(host, port, user, password, db):
    conn = psycopg2.connect(host=host, port=port, user=user, password=password, dbname=db)
    logging.info(f"Database connection established: {db}")
    return conn


# ============================================================================
# SECTION 2: GLOBAL BATCH READING FUNCTIONS
# ============================================================================

def read_global_etl_batch_id(conn):
    try:
        read_query = """
            SELECT * FROM gold_etl_master.gold_etl_global_batch_audit begba 
            WHERE begba.run_status = 'RUNNING'
            ORDER BY batch_run_id DESC, run_start_datetime DESC
        """
        batch_run_id = pd.read_sql_query(read_query, conn)
        if batch_run_id.empty:
            logging.info("No running batch found.")
            return None, None, None, None
        else:
            logging.info(f"Running batch found with batch_run_id: {batch_run_id.iloc[0]['batch_run_id']}")
            return (
                batch_run_id.iloc[0]["batch_run_id"],
                batch_run_id.iloc[0]["run_start_datetime"], 
                batch_run_id.iloc[0]["run_end_datetime"], 
                batch_run_id.iloc[0]["load_type"],
                batch_run_id.iloc[0]["run_frequency"]
            )
    except Exception as e:
        error_msg = f"Error while reading global ETL batch ID: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


# ============================================================================
# SECTION 3: AUDIT LOG FUNCTIONS
# ============================================================================

def update_audit_master_tbl(
    connection,
    batch_run_id,
    etl_name,
    etl_start_time=None,
    etl_end_time=None,
    status=None,
    table_processed=0,
    table_succeeded=0,
    table_failed=0,
    error_msg=None,
    process_id=None,
    etl_schedule_time=None
):
    try:
        cursor = connection.cursor()

        if status and status.lower() == "in_progress":
            if process_id == None:
                cursor.execute("""
                    SELECT COALESCE(MAX(master_audit_id), 0) + 1 AS process_id
                    FROM gold_etl_master.gold_etl_run_master_audit_tbl_test
                """)
                process_id = cursor.fetchone()[0]

            insert_query = """
                INSERT INTO gold_etl_master.gold_etl_run_master_audit_tbl_test (
                    master_audit_id,
                    etl_batch_id,
                    schedule_name,
                    schedule_timestamp,
                    status,
                    start_time,
                    end_time,
                    tables_processed,
                    tables_successful,
                    tables_failed,
                    error_message
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            cursor.execute(insert_query, (
                int(process_id),
                int(batch_run_id),
                etl_name,
                etl_schedule_time,
                status,
                etl_start_time,
                etl_end_time,
                table_processed,
                table_succeeded,
                table_failed,
                error_msg,
            ))
            logging.info(f"New Process Started with Process ID: {process_id}.")
        else:
            if process_id == None:
                raise Exception("Unable to process without Process ID")
            update_query = """
                UPDATE gold_etl_master.gold_etl_run_master_audit_tbl_test
                SET 
                    status = %s,
                    end_time = %s,
                    tables_processed = %s,
                    tables_successful = %s,
                    tables_failed = %s,
                    error_message = %s
                WHERE master_audit_id = %s
            """
            cursor.execute(update_query, (
                status,
                etl_end_time,
                table_processed,
                table_succeeded,
                table_failed,
                error_msg,
                process_id,
            ))
            logging.info(f"Process Completed for Process ID: {process_id}.")
        connection.commit()
        return process_id
    except Exception as e:
        error_msg = f"Failed to update audit master table: {str(e)}"
        logging.error(error_msg)
        raise Exception(error_msg)


def upsert_etl_audit_record(
    connection,
    process_id,
    batch_run_id,
    source_name,
    destination_table,
    operation,
    time_from,
    status,
    input_row=0,
    time_to=None,
    output_row=0,
    error_code=None,
    result=None,
    etl_end_time=None,
    etl_start_time=None
):
    try:
        input_row = int(input_row) if input_row is not None else None
        output_row = int(output_row) if output_row is not None else None

        with connection.cursor() as cur:
            if status.lower() == "in-progress":
                insert_to_audit_tbl_query = """
                    INSERT INTO gold_etl_master.gold_etl_run_table_audit_log (
                        process_id, batch_run_id, data_source, destination_table_name,
                        operation, time_from, time_to, etl_start_time, etl_end_time,
                        input_row, output_row, operation_result, errorcode, result
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """
                cur.execute(insert_to_audit_tbl_query, (
                    int(process_id),
                    int(batch_run_id),
                    source_name,
                    destination_table,
                    operation,
                    time_from,
                    time_to,
                    etl_start_time,
                    time_to,
                    input_row,
                    output_row,
                    status,
                    error_code,
                    result,
                ))
                logging.info(f"ETL Process start logged successfully.")
            else:
                # FIX 1: Corrected 'status=%' typo to 'operation_result = %s,'
                # FIX 2: Removed duplicate 'result' from values tuple (was 11 values, now 10)
                update_audit_tbl_query = """
                    UPDATE gold_etl_master.gold_etl_run_table_audit_log
                    SET time_to = %s,
                        etl_end_time = %s,
                        input_row = %s,
                        output_row = %s,
                        operation_result = %s,
                        errorcode = %s,
                        result = %s
                    WHERE process_id = %s AND batch_run_id = %s AND destination_table_name = %s
                """
                cur.execute(update_audit_tbl_query, (
                    time_to,
                    etl_end_time,
                    int(input_row),
                    int(output_row),
                    status,
                    error_code,
                    result,
                    int(process_id),
                    int(batch_run_id),
                    destination_table
                ))
                logging.info(f"ETL Process Completion logged successfully.")
            connection.commit()
    except Exception as e:
        error_msg = f"Audit log failed: {str(e)}"
        logging.error(error_msg)
        connection.rollback()
        raise Exception(error_msg)


# ============================================================================
# SECTION 4: CONFIGURATION READING FUNCTIONS
# ============================================================================

def get_gold_layer_etl_config(
    connection,
    source=None,
    target_table_type=None,
    target_table_names=None,
    run_type=None,
    run_frequency=None
):
    def safe_literal_eval(value):
        if isinstance(value, (list, tuple)):
            return value
        try:
            return ast.literal_eval(value)
        except (SyntaxError, ValueError, TypeError):
            return value

    query = """
        SELECT *
        FROM gold_etl_master.gold_etl_config_tbl
        WHERE is_config_active = True
    """
    params = []

    if source:
        source = safe_literal_eval(source)
        if isinstance(source, (list, tuple)) and len(source) > 0:
            placeholders = ",".join(["%s"] * len(source))
            query += f" AND source IN ({placeholders})"
            params.extend(source)
        else:
            query += " AND source = %s"
            params.append(source)

    if target_table_type:
        query += " AND target_table_type = %s"
        params.append(target_table_type)

    if target_table_names:
        target_table_names = safe_literal_eval(target_table_names)
        if isinstance(target_table_names, (list, tuple)) and len(target_table_names) > 0:
            placeholders = ",".join(["%s"] * len(target_table_names))
            query += f" AND target_table_name IN ({placeholders})"
            params.extend(target_table_names)
        else:
            query += " AND target_table_name = %s"
            params.append(target_table_names)

    if run_type and run_type.upper() == "DELTA":
        if run_frequency:
            query += " AND run_frequency = %s"
            params.append(run_frequency)
        else:
            logging.warning("run_frequency not provided for DELTA run, including all tables")

    query += " ORDER BY execution_order"
    df = pd.read_sql(query, connection, params=params)
    logging.info(f"Retrieved {len(df)} active configuration records")
    return df


# ============================================================================
# SECTION 5: UTILITY FUNCTIONS
# ============================================================================

def get_last_processed_datetime(gold_connection, table_name):
    query = """
        select
            time_to as last_processed_time
        from
            gold_etl_master.gold_etl_run_table_audit_log getal
        join gold_etl_master.gold_etl_global_batch_audit gegba 
            on gegba.batch_run_id = getal.batch_run_id
        where
            getal.destination_table_name like %s
            and gegba.load_type = 'DELTA'
            and operation_result = 'Success'
        order by
            time_to desc
        limit 1
    """
    try:
        with gold_connection.cursor() as cur:
            cur.execute(query, (table_name,))
            result = cur.fetchone()
            if result and result[0]:
                return result[0]
            else:
                return None
    except Exception as e:
        logging.error(f"Failed to fetch last processed datetime for table {table_name}: {e}")
        return None


def replace_time_bounds_in_query(sql_query, run_type, lower_bound=None, upper_bound=None):
    """
    Replaces time bound placeholders in the SQL query.
    Supports both naming conventions:
        {upper_bound_date} / {upper_bound_time}
        {lower_bound_date} / {lower_bound_time}

    Args:
        sql_query (str): SQL query template.
        run_type (str): Type of ETL run ('DELTA' or 'FULL').
        lower_bound (datetime or str, optional): Lower bound date/time for DELTA runs.
        upper_bound (datetime or str, optional): Upper bound date/time.

    Returns:
        str: SQL query with placeholders replaced.
    """
    # Normalise upper_bound to string once
    if upper_bound:
        if isinstance(upper_bound, datetime):
            upper_bound_str = upper_bound.strftime('%Y-%m-%d %H:%M:%S')
        else:
            upper_bound_str = str(upper_bound)
        # Replace both placeholder variants for upper bound
        for placeholder in ("{upper_bound_date}", "{upper_bound_time}"):
            if placeholder in sql_query:
                sql_query = sql_query.replace(placeholder, f"'{upper_bound_str}'")
                logging.info(f"Upper bound set to: {upper_bound_str}")

    # Replace lower bound based on run type
    if run_type.upper() == "DELTA":
        if lower_bound:
            if isinstance(lower_bound, datetime):
                lower_bound_str = lower_bound.strftime('%Y-%m-%d %H:%M:%S')
            else:
                lower_bound_str = str(lower_bound)
            # Replace both placeholder variants for lower bound
            for placeholder in ("{lower_bound_date}", "{lower_bound_time}"):
                if placeholder in sql_query:
                    sql_query = sql_query.replace(placeholder, f"'{lower_bound_str}'")
                    logging.info(f"Lower bound set to: {lower_bound_str}")
        else:
            for placeholder in ("{lower_bound_date}", "{lower_bound_time}"):
                if placeholder in sql_query:
                    logging.warning(f"Lower bound placeholder '{placeholder}' found but no value provided for DELTA run")

    elif run_type.upper() == "FULL":
        # For FULL runs, replace lower bound placeholders with a default early date
        for placeholder in ("{lower_bound_date}", "{lower_bound_time}"):
            if placeholder in sql_query:
                sql_query = sql_query.replace(placeholder, "'2000-01-01'")
                logging.info("Lower bound set to default (2000-01-01) for FULL run")

    return sql_query


def add_order_by_clause(sql_query, load_key):
    if not load_key:
        return sql_query
    sql_query = re.sub(r';\s*$', '', sql_query)
    if not re.search(r'\border\s+by\b', sql_query, re.IGNORECASE):
        order_by_cols = [col.strip() for col in load_key.split(',')]
        order_by_clause = " ORDER BY " + ", ".join(order_by_cols)
        sql_query += order_by_clause
    return sql_query


def clean_dataframe_for_db(df: pd.DataFrame, use_null_str: bool = False) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            non_null_mask = df[col].notna()
            if non_null_mask.sum() > 0 and df.loc[non_null_mask, col].apply(lambda x: float(x).is_integer()).all():
                df[col] = df[col].astype('Int64')
    if use_null_str:
        df = df.astype(object).where(pd.notna(df), '\\N')
    else:
        df = df.astype(object).where(pd.notna(df), None)
    return df


def generate_incremental_where_clause(incremental_keys, lower_bound_time):
    if not incremental_keys or not lower_bound_time:
        return ""
    if isinstance(lower_bound_time, datetime):
        lower_bound_time = lower_bound_time.strftime('%Y-%m-%d %H:%M:%S')
    conditions = [f'"{key}" > \'{lower_bound_time}\'' for key in incremental_keys]
    where_clause = "WHERE " + " OR ".join(conditions)
    return where_clause


# ============================================================================
# SECTION 6: DATA LOADING FUNCTIONS
# ============================================================================

def load_to_gold(
    df_chunk,
    target_schema_name,
    target_table_name,
    gold_conn,
    ingestion_strategy="TRUNCATE_INSERT",
    conflict_columns=None
):
    if df_chunk.empty:
        logging.debug(f"No data to load for table {target_schema_name}.{target_table_name}")
        return

    df_chunk = clean_dataframe_for_db(df_chunk)
    table_full_name = f'"{target_schema_name}"."{target_table_name}"'

    columns = list(df_chunk.columns)
    insert_cols = ', '.join([f'"{col}"' for col in columns])
    value_placeholders = ', '.join(['%s'] * len(columns))

    if ingestion_strategy in ["TRUNCATE_INSERT", "APPEND"]:
        insert_sql = f'''
            INSERT INTO {table_full_name} ({insert_cols})
            VALUES ({value_placeholders})
        '''
        try:
            with gold_conn.cursor() as cur:
                cur.executemany(insert_sql, df_chunk.values.tolist())
            gold_conn.commit()
            logging.debug(f"Inserted {len(df_chunk)} rows into {table_full_name}")
        except Exception as e:
            logging.exception(f"Failed to insert data into {table_full_name}: {str(e)}")
            gold_conn.rollback()
            raise

    elif ingestion_strategy == "MERGE":
        if not conflict_columns:
            raise ValueError("conflict_columns must be provided for MERGE strategy")

        if isinstance(conflict_columns, str):
            conflict_cols = [col.strip() for col in conflict_columns.split(',')]
        else:
            conflict_cols = conflict_columns

        conflict_clause = ', '.join([f'"{col}"' for col in conflict_cols])
        update_cols = [col for col in columns if col not in conflict_cols]
        update_set = ', '.join([f'"{col}" = EXCLUDED."{col}"' for col in update_cols])

        insert_sql = f'''
            INSERT INTO {table_full_name} ({insert_cols})
            VALUES ({value_placeholders})
            ON CONFLICT ({conflict_clause}) DO UPDATE SET {update_set}
        '''

        try:
            with gold_conn.cursor() as cur:
                cur.executemany(insert_sql, df_chunk.values.tolist())
            gold_conn.commit()
            logging.debug(f"Upserted {len(df_chunk)} rows into {table_full_name}")
        except Exception as e:
            logging.exception(f"Failed to upsert data into {table_full_name}: {str(e)}")
            gold_conn.rollback()
            raise
    else:
        raise ValueError(f"Unknown ingestion_strategy: {ingestion_strategy}")

# ============================================================================
# SECTION 7: TABLE PROCESSING FUNCTIONS
# ============================================================================

def process_table_delta(silver_conn, gold_conn, row, lower_bound_time, upper_bound_time, batch_size=50000):
    sql_query = row['data_load_sql_template']
    sql_query = replace_time_bounds_in_query(sql_query, 'DELTA', lower_bound_time, upper_bound_time)
    logging.info(f"SQL Template: {sql_query}")
    target_table_name = row['target_table_name']
    target_schema_name = row['target_schema_name']
    ingestion_strategy = row['ingestion_strategy']
    load_key = row['load_key']
    incremental_extract_key_str = str(row['incremental_extract_key'])
    incremental_extract_key = [col.strip() for col in incremental_extract_key_str.split(",") if col.strip()]
    total_fetched = 0
    total_processed = 0
    final_status = "failed"
    error_message = None
    offset = 0
    batch_count = 0

    try:
        if ingestion_strategy.upper() == "APPEND" and incremental_extract_key:
            where_clause = generate_incremental_where_clause(incremental_extract_key, lower_bound_time)
            if where_clause:
                delete_query = f'DELETE FROM "{target_schema_name}"."{target_table_name}" {where_clause}'
                with gold_conn.cursor() as cursor:
                    cursor.execute(delete_query)
                    deleted_count = cursor.rowcount
                    gold_conn.commit()
                    logging.info(f"Deleted {deleted_count} existing records from {target_table_name} for incremental load")

        if ingestion_strategy.upper() == "TRUNCATE_INSERT":
            truncate_query = f'TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" RESTART IDENTITY CASCADE'
            with gold_conn.cursor() as cur:
                cur.execute(truncate_query)
            gold_conn.commit()
            logging.info(f"Truncated table: {target_table_name}")

        sql_query_with_order_by = add_order_by_clause(sql_query, load_key)
        sql_query_to_execute = f"{sql_query_with_order_by} LIMIT {batch_size} OFFSET {offset}"

        while True:
            try:
                df_chunk = pd.read_sql_query(sql_query_to_execute, silver_conn)
            except Exception as e:
                error_message = f"Error executing data extraction query: {e}"
                logging.error(error_message)
                raise

            if df_chunk.empty:
                if batch_count == 0:
                    logging.warning(f"No data found for table {target_table_name}")
                    final_status = "success"
                    error_message = "No data found in source table"
                else:
                    logging.debug(f"All batches processed for table {target_table_name}")
                    final_status = "success"
                break

            batch_count += 1
            total_fetched += len(df_chunk)

            try:
                load_to_gold(df_chunk, target_schema_name, target_table_name, gold_conn,
                            ingestion_strategy, conflict_columns=load_key)
                total_processed += len(df_chunk)
                logging.debug(f"Batch {batch_count} processed for table {target_table_name}: {len(df_chunk)} rows")
            except Exception as e:
                error_message = f"Error processing/loading batch {batch_count}: {e}"
                logging.error(error_message)
                raise

            offset += len(df_chunk)
            if len(df_chunk) < batch_size:
                final_status = "success"
                break
            sql_query_to_execute = f"{sql_query_with_order_by} LIMIT {batch_size} OFFSET {offset}"

    except Exception as e:
        if error_message is None:
            error_message = f"Unexpected error: {e}"
        final_status = "failed"
        logging.error(f"Process failed for table {target_table_name}: {error_message}")

    logging.info(f"Table {target_table_name} processed: {batch_count} batches, {total_processed} rows, status: {final_status}")
    return total_fetched, total_processed, final_status, error_message


def process_table_full(silver_conn, gold_conn, row, upper_bound_time, batch_size=50000):
    sql_query = row['data_load_sql_template']
    sql_query = replace_time_bounds_in_query(sql_query, 'FULL', None, upper_bound_time)
    target_table_name = row['target_table_name']
    target_schema_name = row['target_schema_name']
    load_key = row['load_key']
    total_fetched = 0
    total_processed = 0
    final_status = "failed"
    error_message = None
    offset = 0
    batch_count = 0

    try:
        logging.info(f"Truncating target table: {target_table_name}")
        truncate_query = f'TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" RESTART IDENTITY CASCADE'
        with gold_conn.cursor() as cur:
            cur.execute(truncate_query)
            gold_conn.commit()

        sql_query_with_order_by = add_order_by_clause(sql_query, load_key)
        sql_query_to_execute = f"{sql_query_with_order_by} LIMIT {batch_size} OFFSET {offset}"

        while True:
            try:
                df_chunk = pd.read_sql_query(sql_query_to_execute, silver_conn)
            except Exception as e:
                error_message = f"Error executing data extraction query: {e}"
                logging.error(error_message)
                raise

            if df_chunk.empty:
                if batch_count == 0:
                    logging.warning(f"No data found for table {target_table_name}")
                    final_status = "success"
                    error_message = "No data found in source table"
                else:
                    logging.debug(f"All batches processed for table {target_table_name}")
                    final_status = "success"
                break

            batch_count += 1
            total_fetched += len(df_chunk)

            try:
                load_to_gold(df_chunk, target_schema_name, target_table_name, gold_conn,
                            ingestion_strategy="TRUNCATE_INSERT")
                total_processed += len(df_chunk)
                logging.debug(f"Batch {batch_count} processed for table {target_table_name}: {len(df_chunk)} rows")
            except Exception as e:
                error_message = f"Error processing/loading batch {batch_count}: {e}"
                logging.error(error_message)
                raise

            offset += len(df_chunk)
            if len(df_chunk) < batch_size:
                final_status = "success"
                break
            sql_query_to_execute = f"{sql_query_with_order_by} LIMIT {batch_size} OFFSET {offset}"

    except Exception as e:
        if error_message is None:
            error_message = f"Unexpected error: {e}"
        final_status = "failed"
        logging.error(f"Process failed for table {target_table_name}: {error_message}")

    logging.info(f"Table {target_table_name} processed: {batch_count} batches, {total_processed} rows, status: {final_status}")
    return total_fetched, total_processed, final_status, error_message


# ============================================================================
# SECTION 8: SUMMARY LOGGING
# ============================================================================

def log_etl_summary(
    start_time, end_time, total_tables, successful_tables,
    failed_tables, failed_table_names, total_records_fetched, total_records_processed
):
    duration = end_time - start_time
    success_rate = (successful_tables / total_tables * 100) if total_tables > 0 else 0
    logging.info("=" * 70)
    logging.info("ETL RUN SUMMARY")
    logging.info("=" * 70)
    logging.info(f"Start Time:          {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    logging.info(f"End Time:            {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    logging.info(f"Duration:            {duration}")
    logging.info(f"Total Tables:        {total_tables}")
    logging.info(f"Successful Tables:   {successful_tables}")
    logging.info(f"Failed Tables:       {failed_tables}")
    logging.info(f"Success Rate:        {success_rate:.1f}%")
    logging.info(f"Records Fetched:     {total_records_fetched}")
    logging.info(f"Records Processed:   {total_records_processed}")
    if failed_tables > 0:
        logging.info("FAILED TABLES:")
        for i, table_name in enumerate(failed_table_names, 1):
            logging.info(f"  {i}. {table_name}")
    if failed_tables == 0:
        logging.info("STATUS: COMPLETED SUCCESSFULLY")
    else:
        logging.warning(f"STATUS: COMPLETED WITH {failed_tables} FAILURE(S)")
    logging.info("=" * 70)


# ============================================================================
# SECTION 9: MAIN ETL ORCHESTRATION FUNCTION
# ============================================================================

def main(
    dl_host,
    dl_port,
    dl_user,
    dl_pass,
    gold_db_name,
    silver_dbname,
    etl_name="GOLD_UNIFIED_ETL_RUN",
    source_name=None,
    target_table_type=None,
    target_table_names=None,
    batch_size=50000,
):
    etl_run_id = None
    etl_start_time = datetime.now()
    etl_end_time = None
    total_tables = 0
    successful_tables = 0
    failed_tables = 0
    total_records_fetched = 0
    total_records_processed = 0
    failed_table_names = []

    logging.info("=" * 70)
    logging.info("STARTING UNIFIED GOLD LAYER ETL PROCESS")
    logging.info("=" * 70)
    logging.info(f"ETL Name: {etl_name}")

    gold_conn = create_connection(dl_host, dl_port, dl_user, dl_pass, gold_db_name)
    silver_conn = create_connection(dl_host, dl_port, dl_user, dl_pass, silver_dbname)

    try:
        # STEP 1: Read global batch parameters
        logging.info("STEP 1: Reading global ETL batch parameters...")
        batch_run_id, run_start_datetime, run_end_datetime, load_type, run_frequency = read_global_etl_batch_id(gold_conn)

        if batch_run_id is None:
            error_msg = "No running batch found in gold_etl_global_batch_audit table"
            logging.error(error_msg)
            raise Exception(error_msg)

        logging.info(f"Batch Run ID: {batch_run_id}")
        logging.info(f"Load Type: {load_type}")
        logging.info(f"Run Frequency: {run_frequency}")
        logging.info(f"Run End DateTime (Upper Bound): {run_end_datetime}")

        # STEP 2: Validate run_type
        if load_type not in ['DELTA', 'FULL']:
            error_msg = f"Unidentified run type: {load_type}. Expected 'DELTA' or 'FULL'"
            logging.error(error_msg)
            raise Exception(error_msg)

        # STEP 3: Create master audit record
        logging.info("STEP 2: Creating ETL run master audit record...")
        etl_start_time = datetime.now()
        etl_run_id = update_audit_master_tbl(
            connection=gold_conn,
            batch_run_id=batch_run_id,
            etl_name=etl_name,
            etl_start_time=etl_start_time,
            etl_end_time=None,
            status="In_Progress",
            table_processed=0,
            table_succeeded=0,
            table_failed=0,
            error_msg=None,
            etl_schedule_time=etl_start_time
        )
        logging.info(f"ETL Run ID: {etl_run_id}")

        # STEP 4: Read configuration table
        logging.info("STEP 3: Reading ETL configuration table...")
        config_table_df = get_gold_layer_etl_config(
            connection=gold_conn,
            source=source_name,
            target_table_type=target_table_type,
            target_table_names=target_table_names,
            run_type=load_type,
            run_frequency=run_frequency
        )

        if config_table_df.empty:
            logging.warning("No tables found matching the criteria")
            update_audit_master_tbl(
                gold_conn,
                batch_run_id=batch_run_id,
                etl_name=etl_name,
                etl_start_time=None,
                etl_end_time=datetime.now(),
                status="Success",
                table_processed=0,
                table_succeeded=0,
                table_failed=0,
                error_msg="No tables found matching the criteria.",
                process_id=etl_run_id,
            )
            return {
                'status': 'Success',
                'total_tables': 0,
                'successful_tables': 0,
                'failed_tables': 0,
                'message': 'No tables found matching the criteria.'
            }

        config_table_df = config_table_df.sort_values(by="execution_order", ascending=True).reset_index(drop=True)
        logging.info(f"Found {len(config_table_df)} tables to process")

        # STEP 5: Bifurcate based on run_type
        logging.info("=" * 70)
        logging.info(f"STEP 4: Processing tables - Run Type: {load_type}")
        logging.info("=" * 70)

        for _, row in config_table_df.iterrows():
            total_tables += 1
            table_process_start_time = datetime.now()
            table_name = row['target_table_name']
            source_name_current = row['source']
            destination_table = row['target_table_name']
            operation = row['ingestion_strategy']

            logging.info(f"\nProcessing table {total_tables}/{len(config_table_df)}: {table_name}")

            # Create table audit record
            upsert_etl_audit_record(
                gold_conn,
                etl_run_id,
                batch_run_id,
                source_name_current,
                destination_table,
                operation,
                time_from=run_start_datetime,
                input_row=0,
                status="In-Progress",
                time_to=run_end_datetime,
                output_row=0,
                error_code=None,
                result=None,
                etl_start_time=etl_start_time
            )

            total_fetched = 0
            total_processed = 0
            final_status = 'success'
            error_message = None

            try:
                if load_type == 'DELTA':
                    lower_bound_time = get_last_processed_datetime(gold_conn, table_name)

                    # If previous run exists
                    if lower_bound_time:
                        if isinstance(lower_bound_time, str):
                            lower_bound_time = datetime.strptime(lower_bound_time, "%Y-%m-%d %H:%M:%S")

                        lower_bound_time = lower_bound_time - timedelta(minutes=5)
                        logging.info(f"Lower bound time for {table_name}: {lower_bound_time}")

                    # If first run
                    else:
                        lower_bound_time = datetime(2000, 1, 1)
                        logging.info(
                            f"No previous run found for {table_name}, using default lower bound: {lower_bound_time}"
                        )

                    upper_bound_time = run_end_datetime

                    total_fetched, total_processed, final_status, error_message = process_table_delta(
                        silver_conn=silver_conn,
                        gold_conn=gold_conn,
                        row=row,
                        lower_bound_time=lower_bound_time,
                        upper_bound_time=upper_bound_time,
                        batch_size=batch_size
                    )

                elif load_type == 'FULL':
                    upper_bound_time = run_end_datetime
                    total_fetched, total_processed, final_status, error_message = process_table_full(
                        silver_conn=silver_conn,
                        gold_conn=gold_conn,
                        row=row,
                        upper_bound_time=upper_bound_time,
                        batch_size=batch_size
                    )

                total_records_fetched += total_fetched
                total_records_processed += total_processed

            except Exception as table_exc:
                final_status = "failed"
                error_message = f"Exception in processing table: {table_exc}"
                logging.error(f"Table {table_name} failed: {error_message}")
                failed_tables += 1
                failed_table_names.append(table_name)
            else:
                if final_status == "success":
                    successful_tables += 1
                    logging.info(f"Table {table_name} completed successfully")
                else:
                    failed_tables += 1
                    failed_table_names.append(table_name)
                    logging.error(f"Table {table_name} failed")

            # Update table audit log
            upsert_etl_audit_record(
                gold_conn,
                etl_run_id,
                batch_run_id,
                source_name_current,
                destination_table,
                operation,
                time_from=run_start_datetime,
                status="Success" if final_status == "success" else "Failed",
                input_row=total_fetched,
                time_to=run_end_datetime,
                output_row=total_processed,
                error_code=error_message,
                result="Success" if final_status == "success" else "Failed",
                etl_end_time=datetime.now(),
                etl_start_time=etl_start_time
            )

        # STEP 6: Log summary and update master audit
        etl_end_time = datetime.now()
        log_etl_summary(
            etl_start_time, etl_end_time, total_tables, successful_tables,
            failed_tables, failed_table_names, total_records_fetched, total_records_processed
        )

        update_audit_master_tbl(
            connection=gold_conn,
            batch_run_id=batch_run_id,
            etl_name=etl_name,
            etl_start_time=None,
            etl_end_time=etl_end_time,
            status="Success" if failed_tables == 0 else "Failed",
            table_processed=total_tables,
            table_succeeded=successful_tables,
            table_failed=failed_tables,
            error_msg=None if failed_tables == 0 else f"Failed tables: {', '.join(failed_table_names)}",
            process_id=etl_run_id
        )

    except Exception as main_exc:
        etl_end_time = datetime.now()
        logging.error(f"ETL main process failed: {main_exc}")

        if etl_run_id:
            update_audit_master_tbl(
                connection=gold_conn,
                batch_run_id=batch_run_id,
                etl_name=etl_name,
                etl_start_time=None,
                etl_end_time=etl_end_time,
                status="Failed",
                table_processed=total_tables,
                table_succeeded=successful_tables,
                table_failed=failed_tables,
                error_msg=str(main_exc),
                process_id=etl_run_id
            )

        if total_tables > 0:
            log_etl_summary(
                etl_start_time, etl_end_time, total_tables, successful_tables,
                failed_tables, failed_table_names, total_records_fetched, total_records_processed
            )
        raise

    finally:
        gold_conn.close()
        silver_conn.close()
        logging.info("Database connections closed")
        logging.info("=" * 70)
        logging.info("ETL PROCESS COMPLETED")
        logging.info("=" * 70)

In [ ]:
# Credentials:
host="host"
port='port'
user="user"
password="password"
silver_db="silver_db"
gold_db="gold_db"

In [ ]:
main(
    dl_host=host,
    dl_port=port,
    dl_user=user,
    dl_pass=password,
    gold_db_name=gold_db,
    silver_dbname=silver_db,
    etl_name="GOLD_ETL_FULL_RUN",
    source_name=('ExistingMB',),
    target_table_type=None,
    target_table_names=('dm_service_fact',),
    batch_size=100000,
)